# Heart Disease Prediction using Machine Learning

A machine learning project that predicts whether a patient has heart disease using five classification algorithms.

- `0` = Patient does NOT have heart disease
- `1` = Patient DOES have heart disease

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

## 2. Generate Dataset (10,000 Patients)

We generate a synthetic dataset of 10,000 patients using clinically-motivated risk scoring logic.

In [ ]:
np.random.seed(42)
n = 10000

age      = np.random.randint(30, 80, n)
sex      = np.random.randint(0, 2, n)
cp       = np.random.randint(0, 4, n)
trestbps = np.random.randint(90, 200, n)
chol     = np.random.randint(150, 400, n)
fbs      = np.random.randint(0, 2, n)
restecg  = np.random.randint(0, 3, n)
thalach  = np.random.randint(70, 210, n)
exang    = np.random.randint(0, 2, n)
oldpeak  = np.round(np.random.uniform(0, 6, n), 1)
slope    = np.random.randint(0, 3, n)
ca       = np.random.randint(0, 4, n)
thal     = np.random.randint(0, 4, n)

risk_score = (
    (age > 55).astype(int) * 2 +
    (sex == 1).astype(int) * 1 +
    (cp >= 2).astype(int) * 2 +
    (trestbps > 140).astype(int) * 1 +
    (chol > 240).astype(int) * 1 +
    (fbs == 1).astype(int) * 1 +
    (thalach < 120).astype(int) * 2 +
    (exang == 1).astype(int) * 2 +
    (oldpeak > 2).astype(int) * 2 +
    (ca > 0).astype(int) * 2
)

noise  = np.random.randint(0, 3, n)
target = ((risk_score + noise) >= 8).astype(int)

df = pd.DataFrame({
    'age': age, 'sex': sex, 'cp': cp, 'trestbps': trestbps,
    'chol': chol, 'fbs': fbs, 'restecg': restecg, 'thalach': thalach,
    'exang': exang, 'oldpeak': oldpeak, 'slope': slope,
    'ca': ca, 'thal': thal, 'target': target
})

print(f'Dataset shape: {df.shape}')
print(f'Patients WITH heart disease    : {(df["target"] == 1).sum()}')
print(f'Patients WITHOUT heart disease : {(df["target"] == 0).sum()}')
df.head()

## 3. Data Preprocessing & EDA
**Lead: Data Preprocessing & EDA**

This section covers:
- Handling missing values
- Outlier detection
- Exploratory Data Analysis (EDA)
- Feature Encoding

In [ ]:
# --- Missing Values ---
print('Missing values per column:')
print(df.isnull().sum())
print(f'Total missing: {df.isnull().sum().sum()}')

In [ ]:
# --- Outlier Detection using IQR ---
print('Outlier Detection (IQR method):')
for col in ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    print(f'  {col:<12}: {n_out} outliers')

In [ ]:
# --- EDA: Basic Statistics ---
print('Basic Statistics:')
df.describe()

In [ ]:
# --- EDA: Correlation with Target ---
print('Feature correlation with target (sorted):')
corr = df.corr()['target'].drop('target').sort_values(ascending=False)
print(corr)

In [ ]:
# --- Feature Encoding ---
# All features are already integer-encoded; no text-to-number conversion needed.
# Binary  : sex (0/1), fbs (0/1), exang (0/1)
# Ordinal : cp (0-3), restecg (0-2), slope (0-2), thal (0-3), ca (0-3)
print('Feature Encoding:')
print('  All categorical features are already integer-encoded.')
print('  Binary : sex, fbs, exang')
print('  Ordinal: cp, restecg, slope, thal, ca')
print('  No additional encoding required.')

## 4. Split Data (80% Train | 20% Test)
**Lead: Modeling Lead - Baseline Algorithms**

`test_size=0.20` → 8,000 training samples and 2,000 test samples.

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'Training samples : {X_train.shape[0]}  (80%)')
print(f'Testing  samples : {X_test.shape[0]}   (20%)')

## 5. Feature Scaling (Standardization)
**Lead: Data Preprocessing & EDA**

We apply `StandardScaler` (mean=0, std=1). The scaler is fitted **only** on training data to prevent data leakage, then applied to both train and test sets.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit on train only
X_test_scaled  = scaler.transform(X_test)        # apply same params to test

print('Feature scaling done! (StandardScaler — mean=0, std=1)')

## 6. Evaluation Helper
**Lead: Evaluation & Reporting**

A shared helper that computes Accuracy, Precision, Recall, F1-Score, and the Confusion Matrix for any model.

In [ ]:
results = []

def evaluate_model(name, y_test, y_pred):
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    cm   = confusion_matrix(y_test, y_pred)

    print(f"\n{'='*50}")
    print(f'  {name}')
    print(f"{'='*50}")
    print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1-Score  : {f1:.4f}')
    print(f'\n  Confusion Matrix:')
    print(f'    TN={cm[0,0]}  FP={cm[0,1]}')
    print(f'    FN={cm[1,0]}  TP={cm[1,1]}')
    print()
    print(classification_report(y_test, y_pred, target_names=['No Disease', 'Has Disease']))

    return {'Model': name, 'Accuracy': round(acc, 4), 'Precision': round(prec, 4),
            'Recall': round(rec, 4), 'F1-Score': round(f1, 4)}

## 7. Baseline Models
**Lead: Modeling Lead - Baseline Algorithms**

We start with two simple, interpretable baselines before moving to more complex models.

### Model 1 — Logistic Regression
Draws a linear decision boundary to separate patients. Fast and interpretable, but limited to linearly separable problems.

In [ ]:
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)

results.append(evaluate_model('Logistic Regression', y_test, lr_pred))

### Model 2 — Naive Bayes
Uses Bayes' theorem, assuming all features are independent. Very fast but struggles when features are correlated (e.g. age & thalach).

In [ ]:
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)
nb_pred = nb_model.predict(X_test_scaled)

results.append(evaluate_model('Naive Bayes', y_test, nb_pred))

In [ ]:
# Baseline Analysis — where do these models fail?
print('Baseline Analysis:')
print('  Logistic Regression assumes a linear decision boundary.')
print('  It will miss non-linear patterns present in oldpeak, thalach, ca.')
print()
print('  Naive Bayes assumes feature independence.')
print('  It underperforms when correlated features carry joint predictive power.')

## 8. Advanced Models + Hyperparameter Tuning
**Lead: Modeling Lead - Advanced Algorithms**

We implement distance-based (KNN), margin-based (SVM), and ensemble (Random Forest) models with hyperparameter tuning.

### Model 3 — KNN (Hyperparameter Tuning: K selection)
Classifies a patient by the majority label of its K nearest neighbours. We try K = 3, 5, 7, 9, 11 and pick the best.

In [ ]:
print('KNN — trying K values: 3, 5, 7, 9, 11 ...')
best_k     = 3
best_k_acc = 0

for k in [3, 5, 7, 9, 11]:
    knn_tmp = KNeighborsClassifier(n_neighbors=k)
    knn_tmp.fit(X_train_scaled, y_train)
    acc_tmp = accuracy_score(y_test, knn_tmp.predict(X_test_scaled))
    print(f'  K={k:<2}  Accuracy={acc_tmp:.4f}')
    if acc_tmp > best_k_acc:
        best_k_acc = acc_tmp
        best_k     = k

print(f'\nBest K = {best_k}  (Accuracy: {best_k_acc:.4f})')

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train_scaled, y_train)
knn_pred  = knn_model.predict(X_test_scaled)

results.append(evaluate_model(f'KNN (K={best_k})', y_test, knn_pred))

### Model 4 — SVM
Finds the hyperplane that maximises the margin between classes. We use a linear kernel.

In [ ]:
svm_model = SVC(kernel='linear', C=1, random_state=42)
svm_model.fit(X_train_scaled, y_train)
svm_pred  = svm_model.predict(X_test_scaled)

results.append(evaluate_model('SVM', y_test, svm_pred))

### Model 5 — Random Forest (Hyperparameter Tuning: n_estimators & max_depth)
Builds an ensemble of decision trees and combines their votes. We tune the number of trees and tree depth.

In [ ]:
print('Random Forest — tuning n_estimators and max_depth ...')
rf_candidates = [(50, 5), (100, 5), (100, 10), (150, 10)]
best_rf       = None
best_rf_label = ''
best_rf_acc   = 0

for n_est, depth in rf_candidates:
    rf_tmp = RandomForestClassifier(n_estimators=n_est, max_depth=depth, random_state=42)
    rf_tmp.fit(X_train_scaled, y_train)
    acc_tmp = accuracy_score(y_test, rf_tmp.predict(X_test_scaled))
    print(f'  n_estimators={n_est:<3}  max_depth={depth:<3}  Accuracy={acc_tmp:.4f}')
    if acc_tmp > best_rf_acc:
        best_rf_acc   = acc_tmp
        best_rf       = rf_tmp
        best_rf_label = f'Random Forest (n={n_est}, depth={depth})'

print(f'\nBest: {best_rf_label}  (Accuracy: {best_rf_acc:.4f})')

In [ ]:
rf_pred = best_rf.predict(X_test_scaled)
results.append(evaluate_model(best_rf_label, y_test, rf_pred))

## 9. Final Comparative Report
**Lead: Evaluation & Reporting**

Gathering predictions from all five models and presenting Accuracy, Precision, Recall, and F1-Score side by side.

In [ ]:
comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values('Accuracy', ascending=False).reset_index(drop=True)
comparison_df.index += 1  # rank starts at 1

print('\n' + '=' * 70)
print('                  FINAL MODEL COMPARISON REPORT')
print('=' * 70)
header = f"  {'Rank':<5} {'Model':<35} {'Accuracy':>9} {'Precision':>10} {'Recall':>7} {'F1':>7}"
print(header)
print('  ' + '-' * 66)
for rank, row in comparison_df.iterrows():
    print(f"  {rank:<5} {row['Model']:<35} {row['Accuracy']:>9.4f} "
          f"{row['Precision']:>10.4f} {row['Recall']:>7.4f} {row['F1-Score']:>7.4f}")
print('=' * 70)
print()
print(comparison_df)

## 10. Best Model & Final Prediction

In [ ]:
best_name = comparison_df.iloc[0]['Model']
print(f'Best Model : {best_name}')
print(f'Accuracy   : {comparison_df.iloc[0]["Accuracy"] * 100:.2f}%')

model_map = {
    'Logistic Regression': lr_model,
    'Naive Bayes':         nb_model,
    'SVM':                 svm_model,
}
if best_name not in model_map:
    if best_name.startswith('KNN'):
        model_map[best_name] = knn_model
    elif best_name.startswith('Random Forest'):
        model_map[best_name] = best_rf

best_model = model_map[best_name]

In [ ]:
# Sample patient prediction
sample_patient = pd.DataFrame([{
    'age': 63, 'sex': 1, 'cp': 3, 'trestbps': 145, 'chol': 233,
    'fbs': 1, 'restecg': 0, 'thalach': 150, 'exang': 0,
    'oldpeak': 2.3, 'slope': 0, 'ca': 0, 'thal': 1
}])

sample_scaled = scaler.transform(sample_patient)
prediction    = best_model.predict(sample_scaled)[0]

print('--- Sample Patient Prediction ---')
if prediction == 1:
    print('Patient is likely to HAVE heart disease')
else:
    print('Patient is UNLIKELY to have heart disease')

print('\n0 = No heart disease')
print('1 = Has heart disease')

## 11. Conclusion

In this project we:
- Generated and preprocessed a 10,000-patient synthetic heart disease dataset
- Detected missing values and outliers, and confirmed all features are integer-encoded
- Performed EDA including correlation analysis and basic statistics
- Split the data: **80% training / 20% testing** and applied StandardScaler
- Trained two **baseline models**: Logistic Regression and Naive Bayes, and documented their limitations
- Trained three **advanced models**: KNN (with K tuning), SVM, and Random Forest (with n_estimators & depth tuning)
- Compared all five models using Accuracy, Precision, Recall, F1-Score, and Confusion Matrices
- Selected the best model and used it to predict heart disease for a sample patient

**Prediction meaning:**
- `0` = Patient does NOT have heart disease  
- `1` = Patient DOES have heart disease